# PA3 Task 2 on Kaggle — SigLIP–Qwen VLM: data, features, 3-stage training, evaluation

Runs the whole Task 2 pipeline: select the COCO / VQAv2 examples (seed 6304), download only those images,
cache SigLIP features, train the main 196-token model on GPU 0 and the optional 49-token model on GPU 1
at the same time, evaluate both on the final-test suites, and build the tables and figures.

**Before running:** right-hand panel → *Session options* → **Accelerator: GPU T4 x2**, **Internet: On**. No inputs needed.
Then **Save Version → Save & Run All (Commit)** (about 3–4 h). Close the tab if you like; stop the editor session afterwards.

Afterwards: saved version → **Output** → download `task2_results.zip` (small: logs, predictions, tables, figures)
and unzip it into the repo root (it contains `outputs/task2/...`).

In [ ]:
RUN_POOL49 = True   # optional 49-visual-token run on the second GPU (set False to skip)
!git clone -q https://github.com/ahmadhassann96/ATML_PA3.git /kaggle/working/ATML_PA3
%cd /kaggle/working/ATML_PA3
!git log --oneline -1
!pip install -q "transformers==4.57.1"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import os
os.environ["PA3_OUT"] = "/kaggle/working/outputs"     # small results (saved with the notebook)
os.environ["PA3_T2_DATA"] = "/tmp/task2data"          # raw annotations + images (not saved)
os.environ["PA3_T2_CACHE"] = "/tmp/task2cache"        # SigLIP feature cache, ~3 GB (not saved)
os.environ["HF_HUB_DISABLE_XET"] = "1"
!mkdir -p /tmp/task2data/raw /tmp/task2data/raw_vqa
!cd /tmp/task2data/raw && wget -q http://images.cocodataset.org/annotations/annotations_trainval2014.zip && unzip -q annotations_trainval2014.zip && ls annotations
!cd /tmp/task2data/raw_vqa && for f in v2_Questions_Train_mscoco v2_Questions_Val_mscoco v2_Annotations_Train_mscoco v2_Annotations_Val_mscoco v2_Complementary_Pairs_Train_mscoco v2_Complementary_Pairs_Val_mscoco; do wget -q https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/$f.zip && unzip -q $f.zip; done; ls

In [ ]:
# select every example (seed 6304), then download only the selected COCO images
!python -m task2.data --select
!python -m task2.data --download
!cat /kaggle/working/outputs/task2/data/data_checks.json | head -60

In [ ]:
# cache SigLIP features (FP16), then download the language model once before the two training workers start
!python -m task2.features
from huggingface_hub import snapshot_download
print(snapshot_download("Qwen/Qwen3-0.6B"))
!python -m task2.analyze --labels_only

In [ ]:
# train: main run (196 tokens) on GPU 0, optional 49-token run on GPU 1; logs in /kaggle/working/task2_<run>.log
import subprocess, sys, threading, time

def run(gpu, args, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    with open(log, "a") as f:
        rc = subprocess.run([sys.executable, "-m"] + args, stdout=f, stderr=subprocess.STDOUT, env=env).returncode
    print(f"{' '.join(args)} finished with exit code {rc}", flush=True)

jobs = [(0, ["task2.train", "--run", "main"], "main")]
if RUN_POOL49:
    jobs.append((1, ["task2.train", "--run", "pool49", "--pool", "2"], "pool49"))
threads = [threading.Thread(target=run, args=(g, a, f"/kaggle/working/task2_{n}.log")) for g, a, n in jobs]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    time.sleep(300)
    for _, _, n in jobs:
        p = f"/kaggle/working/task2_{n}.log"
        if os.path.exists(p):
            lines = [l for l in open(p, errors="ignore").read().splitlines() if l.startswith("[") or "dev@" in l]
            print(time.strftime("%H:%M"), n, lines[-1][:170] if lines else "", flush=True)
for t in threads:
    t.join()
!grep -h "dev@\|finished\|gradient check\|Error\|error" /kaggle/working/task2_*.log | tail -40

In [ ]:
# final evaluation (both runs in parallel), then tables and figures
jobs = [(0, ["task2.evaluate", "--run", "main"], "eval_main")]
if RUN_POOL49:
    jobs.append((1, ["task2.evaluate", "--run", "pool49"], "eval_pool49"))
threads = [threading.Thread(target=run, args=(g, a, f"/kaggle/working/task2_{n}.log")) for g, a, n in jobs]
for t in threads:
    t.start()
for t in threads:
    t.join()
!cat /kaggle/working/task2_eval_*.log | grep -v Warning
!python -m task2.analyze

In [ ]:
# small results only (no checkpoints / weights); stage exports stay in the notebook output folder
!cd /kaggle/working && zip -qr task2_results.zip outputs/task2 -x "*.pt" "*/checkpoints/*" && cp task2_*.log outputs/task2/ 2>/dev/null; zip -qj task2_results.zip /kaggle/working/task2_*.log
!ls -lh /kaggle/working/task2_results.zip